# 04 - Sensitivity Analysis (No AD)

This notebook reruns core models after filtering to no/low AD pathology (`niareagansc > 2`).

## Script equivalent

The same workflow is available via `scripts/04_sensitivity_no_ad.py`.


In [ ]:
import pandas as pd

from config import DATA_PROCESSED_DIR, FINAL_FORMATTED_FILENAME, TABLES_DIR, ensure_project_dirs
from src.data_utils import get_lipid_columns
from src.stats_utils import (
    RegressionSpec,
    compute_category_means,
    filter_no_ad,
    run_per_lipid_regression,
    split_by_sex,
    zscore_columns,
)

ensure_project_dirs()


## Step 1: Filter dataset to no-AD cohort


In [ ]:
df = pd.read_csv(DATA_PROCESSED_DIR / FINAL_FORMATTED_FILENAME)
no_ad_df = filter_no_ad(df, reagan_column="niareagansc", threshold=2)
no_ad_df.to_csv(TABLES_DIR / "sensitivity_noad_dataset.csv", index=False)
print("Original rows:", len(df))
print("No-AD rows:", len(no_ad_df))


## Step 2: Refit per-lipid and category models by cohort


In [ ]:
lipid_cols = get_lipid_columns(no_ad_df)
spec = RegressionSpec(
    predictors=("SI_avg", "niareagansc", "age_death"),
    primary_predictor="SI_avg",
    min_n=20,
)

cohorts = split_by_sex(no_ad_df)
for cohort_name, cohort_df in cohorts.items():
    lipid_results = run_per_lipid_regression(cohort_df, lipid_columns=lipid_cols, spec=spec)
    lipid_results.to_csv(TABLES_DIR / f"sensitivity_noad_lipid_{cohort_name}.csv", index=False)

    category_df = compute_category_means(cohort_df, lipid_columns=lipid_cols)
    category_cols = [c for c in category_df.columns if c.startswith("catmean_")]
    category_results = run_per_lipid_regression(category_df, lipid_columns=category_cols, spec=spec)
    category_results.to_csv(TABLES_DIR / f"sensitivity_noad_category_{cohort_name}.csv", index=False)

print("Saved sensitivity_noad_lipid_* and sensitivity_noad_category_* tables.")


## Step 3: Scaling sensitivity (z-score lipid features)


In [ ]:
for cohort_name, cohort_df in cohorts.items():
    z_df = zscore_columns(cohort_df, lipid_cols)
    z_results = run_per_lipid_regression(z_df, lipid_columns=lipid_cols, spec=spec)
    z_results.to_csv(TABLES_DIR / f"sensitivity_noad_lipid_zscore_{cohort_name}.csv", index=False)

print("Saved sensitivity_noad_lipid_zscore_* tables.")


## Next notebook

Run `notebooks/05_visualization.ipynb` to regenerate figures from the updated result tables.